# Standard DPO — Sycophancy Resistance Training

Trains **Qwen2-VL-7B-Instruct** with standard DPO on the sycophancy dataset.  
Checkpoints are saved to Google Drive. W&B gives a live loss curve link.

> **Runtime**: Runtime → Change runtime type → **A100 GPU**  
> **Parallel run**: Upload `colab_weighted_dpo.ipynb` to a second Colab tab and run it simultaneously.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_SAVE_DIR = '/content/drive/MyDrive/sycophancy_dpo/standard'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_SAVE_DIR}')

## 2. Install Dependencies

In [ ]:
%%capture install_output
!pip install \
    transformers==4.45.2 \
    trl==0.11.4 \
    peft==0.12.0 \
    accelerate==0.34.2 \
    bitsandbytes==0.43.3 \
    wandb \
    datasets \
    matplotlib -q
print('Installation complete.')

## 3. Clone Repo & Verify Dataset

In [ ]:
import os

REPO_URL = 'https://github.com/rupalirajesh/sycophancy_VidLMs.git'
REPO_DIR = '/content/sycophancy_VidLMs'

# If repo already exists (e.g. reconnected session), just pull latest
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print('Repo ready at:', REPO_DIR)

In [ ]:
import json, os

TRAIN_PATH = f'{REPO_DIR}/output/train.jsonl'
VAL_PATH   = f'{REPO_DIR}/output/val.jsonl'

for path in [TRAIN_PATH, VAL_PATH]:
    size_mb = os.path.getsize(path) / 1e6
    with open(path) as f:
        n = sum(1 for _ in f)
    print(f'{os.path.basename(path)}: {n:,} examples  ({size_mb:.1f} MB)')

# Sanity-check one example
with open(TRAIN_PATH) as f:
    ex = json.loads(f.readline())
print('\nSample fields:', list(ex.keys()))
print('Example type:', ex['example_type'], '| pressure:', ex['pressure_level'], '| weight:', ex['loss_weight'])

## 4. W&B Login
Get your API key from https://wandb.ai/authorize

In [ ]:
import wandb
wandb.login()  # paste API key when prompted

## 5. Configure Output Paths

In [ ]:
CHECKPOINT_DIR = f'{REPO_DIR}/checkpoints/standard-dpo-qwen2vl'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print('Training checkpoints →', CHECKPOINT_DIR)
print('Google Drive backup  →', DRIVE_SAVE_DIR)

# GPU info
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

## 6. Train — Standard DPO

Expected time: ~4–6 hours on A100.  
Click the W&B link printed below to watch live loss curves.

In [ ]:
!python train_dpo.py \
    --train {TRAIN_PATH} \
    --val   {VAL_PATH} \
    --output-dir {REPO_DIR}/checkpoints \
    --epochs 2 \
    --batch-size 2 \
    --grad-accum 8 \
    --lr 5e-5 \
    --beta 0.1 \
    --max-length 1024

## 7. Save to Google Drive

In [ ]:
import shutil, os

print('Copying checkpoints to Drive...')
if os.path.isdir(DRIVE_SAVE_DIR):
    shutil.rmtree(DRIVE_SAVE_DIR)
shutil.copytree(CHECKPOINT_DIR, DRIVE_SAVE_DIR)

# Report what was saved
total = 0
for root, dirs, files in os.walk(DRIVE_SAVE_DIR):
    for f in files:
        total += os.path.getsize(os.path.join(root, f))
print(f'Saved {total/1e9:.2f} GB to {DRIVE_SAVE_DIR}')

## 8. Plot Loss Curve

In [ ]:
import json, matplotlib.pyplot as plt, matplotlib.ticker as mticker

log_path = f'{CHECKPOINT_DIR}/loss_log.json'

with open(log_path) as f:
    entries = json.load(f)

train_steps = [e['step'] for e in entries if 'train_loss' in e]
train_loss  = [e['train_loss'] for e in entries if 'train_loss' in e]
eval_steps  = [e['step'] for e in entries if 'eval_loss' in e]
eval_loss   = [e['eval_loss']  for e in entries if 'eval_loss' in e]
rm_steps    = [e['step'] for e in entries if 'reward_margin' in e]
rm_vals     = [e['reward_margin'] for e in entries if 'reward_margin' in e]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Standard DPO — Training Progress', fontsize=13, fontweight='bold')

axes[0].plot(train_steps, train_loss, color='#2196F3', linewidth=1.5, label='train')
axes[0].set_title('Train Loss'); axes[0].set_xlabel('Step'); axes[0].grid(alpha=0.3)

axes[1].plot(eval_steps, eval_loss, color='#F44336', linewidth=1.5, linestyle='--', label='eval')
axes[1].set_title('Eval Loss'); axes[1].set_xlabel('Step'); axes[1].grid(alpha=0.3)

axes[2].plot(rm_steps, rm_vals, color='#4CAF50', linewidth=1.5)
axes[2].set_title('Reward Margin (chosen − rejected)'); axes[2].set_xlabel('Step'); axes[2].grid(alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plot_path = f'{DRIVE_SAVE_DIR}/loss_curve.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f'Plot saved to {plot_path}')
plt.show()